In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-4: Case study: training a reasoning model

## Where this session fits

> **The limitation we start from: RLHF and RLAIF rewards are learned proxies, and a proxy can be exploited. For some tasks a program can check the answer.**

- **The question:** how is a reasoning model trained end to end with verifiable rewards?
- **Position: session 4 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-2 and RLLM-3 built rewards from comparisons
    - this session uses a reward a program checks, with plain GRPO from Intro session C2 (GRPO)
    - next: RLLM-5 on PPO, RLLM-6 on GRPO and its refinements
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**You are here**

<img src="images/series_map_4.png" width=85%>

**By the end you can**

1. say what a reasoning model does differently, and why that behavior is instilled in post-training
2. say why reasoning RL may start from a short SFT step (supervised fine-tuning), and read the stages of the DeepSeek-R1 recipe
3. compute a rule-based reward and its group advantage for a group of completions
4. say why a rule-based reward was preferred to a learned one there, and what rules cannot do
5. tell an outcome reward from a process reward


# What a reasoning model does

**Why this section?** The case study trains one behavior: thinking before answering.

> **A reasoning model writes a long chain of thought, with reflection and revision, before it gives its answer.**

- the **chain of thought** is step-by-step reasoning in the completion
- **reflection:** looking back at the work so far, and judging the solution and the strategy
- **revision:** changing the current approach
- the completion has two marked parts, so a program can find the answer
    - the reasoning, between `<think>` and `</think>`
    - the answer, between `<answer>` and `</answer>`
    - the markers also let the user hide the reasoning
- this is not the natural behavior of a pretrained model: it is instilled in post-training

**A toy example** (real tasks are much harder; this one needs no long reasoning)

- prompt: "Find a number that, multiplied by 3 and then increased by 7, equals 22."
- a base model's plain completion: "The answer is 8" (a guess, no work shown, wrong)
- a reasoning completion
    - `<think>` let the number be x, so 3x + 7 = 22
    - subtract 7: 3x = 15, then divide by 3: x = 5
    - reflection: "check: 3 times 5 is 15, and 15 plus 7 is 22"
    - revision, when a check fails: "that does not match; try another route"
    - `</think>` then `<answer>5</answer>`


## Why reasoning RL often starts from SFT: the cold start

**Why this section?** RL raises the probability of what the model already does sometimes (Intro session C1). So what if it never does this?

> **The DeepSeek-R1 paper's stated reason for a short SFT step first: R1-Zero, trained without it, produced completions that were hard to read and mixed languages.**

- **R1-Zero:** RL applied straight to the base model with no SFT
    - the base model was DeepSeek-V3-Base, **V3** for short, a very large mixture-of-experts model (only its size matters here)
    - it learned to reason, but its completions were hard to read
- **cold-start SFT:** a short SFT step on examples in the new style
    - it gives a readable start (a human-friendly style)
    - it does not guarantee correct reasoning

**A second reason, our illustration: a model that never succeeds**

> **If no sampled completion earns any reward, there is nothing to learn from.**

- **our reading:** suppose a completion is rewarded with chance $p = 0.01$
    - a group of $G = 8$ contains a rewarded completion with probability $1 - (1-p)^G$, about $8\%$
- a weak base model would need SFT to raise $p$ before RL can start
- a strong base such as V3 follows prompts, so $p$ may be much higher
    - for it, the paper's reason (readability) is the one that applies


## The reward: rules, not a learned model

**Why this section?** This is the point of the case study: the reward is a program.

> **The reward is rule-based: a sum of checks on the completion, computed by a program with nothing to train.**

- an **accuracy** reward: is the final answer correct?
    - a **verifiable** question: we may not know how to find the answer, but given it we can check it
    - for example: a number in a math problem; the same idea checks code with tests
- a **format** reward: are the `<think>` and `<answer>` markers present and in order?
    - without them the answer cannot be found in the completion
- other terms are possible: a reward for staying in one language (used by R1, see the recipe)
- why not a learned reward model (RLLM-2)?
    - the DeepSeek-R1 paper reports that a neural reward model may suffer from reward hacking in large-scale RL (the over-optimization of RLLM-2)
    - our inference: a program has no learned parameters to drift, and needs no labeler for each completion
    - but it needs an answer key for each prompt, and it checks only what its author wrote: rules can be gamed too


## A worked group: rewards and advantages

**Why this section?** The group advantage of Intro session C2, applied to a rule-based reward.

> **Rewards are centered and scaled within the group, so a completion is judged against its siblings, not against a fixed bar.**

- one prompt, $G = 5$ completions: accuracy and format both count 1
    - illustrative values, not the paper's
| completion | what happened | accuracy | format | total reward | advantage |
|:--|:--|:--|:--|:--|:--|
| 1 | correct, markers present | 1 | 1 | 2 | +0.73 |
| 2 | correct, markers present | 1 | 1 | 2 | +0.73 |
| 3 | wrong answer, markers present | 0 | 1 | 1 | -1.10 |
| 4 | correct, but no think markers | 1 | 0 | 1 | -1.10 |
| 5 | correct, markers present | 1 | 1 | 2 | +0.73 |

- the mean of the total rewards is $1.60$ and their standard deviation is $0.55$ (divisor $G - 1$, as in C2)
- the advantage is $(\text{total reward} - \text{mean}) / \text{std}$, and every token of the completion gets it
- implementations differ: with divisor $G$ the advantages are $+0.82$ and $-1.22$ instead of $+0.73$ and $-1.10$

**Reading the table**

- completions 3 and 4 have total reward 1, below the mean 1.6, so both are pushed down
    - completion 3 has the wrong answer
    - completion 4 is correct but lacks the markers, which cost a point
- completions 1, 2 and 5 are pushed up equally: the rule cannot tell them apart
- all totals are non-negative, yet two advantages are negative: centering makes the comparison relative
- if every completion earned the same total, every advantage would be $0$, and nothing would be learned (C2)


# The DeepSeek-R1 recipe, stage by stage

**Why this section?** One published recipe, seen as a sequence of the steps of RLLM-1.

> **Reasoning RL is the middle of a pipeline: SFT before it, more SFT and a second RL stage after it, and distillation at the end.**

<img src="images/rllm4_r1.png" width=85%>

- **reasoning RL:** GRPO with accuracy and format rewards
    - plus a language-consistency reward, against the language mixing of R1-Zero
- **RL for helpfulness and harmlessness:** learned reward models for general prompts, rules for reasoning prompts
- **distillation:** smaller models are fine-tuned on its completions: imitation (RLLM-1) of a model


## Outcome rewards and process rewards

**Why this section?** A reward for the whole completion is coarse. A reward for each step would give credit.

> **An outcome reward scores the whole completion; a process reward scores each step.**

- an **outcome reward** gives one reward at the end: right or wrong
    - it can come from a program or from a learned outcome reward model (ORM)
    - what the rule-based reward of this session does
- a **process reward model** (PRM) gives a reward for each reasoning step
    - for example, step 3 of a solution is wrong: only that step is marked down
    - it needs a label for every step, which is costly
- the DeepSeek-R1 paper reports not using a PRM for its reasoning RL
    - it lists reward hacking of a learned PRM among its reasons
- GRPO's advantage is the same for every token: it uses an outcome reward
    - per-token credit is a theme of RLLM-5 and RLLM-6


# What we established, and what it sets up

- **Established**
    - a reasoning model thinks in a long chain of thought, with reflection and revision
    - the reward is a program: accuracy and format checks, with no model to train
    - rewards are centered and scaled within the group (C2) to give advantages
    - R1-Zero was hard to read, so the recipe starts with a short SFT step
    - a base model that never succeeds would need one too
    - the R1 recipe: cold-start SFT, reasoning RL, rejection sampling and SFT, RL for helpfulness and harmlessness, distillation
    - an outcome reward is coarse
    - a process reward gives credit per step but needs step labels
- **Sets up**
    - **the limits:** rules need verifiable tasks and an answer key, and can be gamed
    - we used plain GRPO and never showed it failing
    - next session, RLLM-5: PPO in depth
    - then RLLM-6: GRPO and its refinements, the weak spots of plain GRPO
